In [3]:
import pandas as pd
import json
import glob

# cargar todos los json de la carpeta
files = glob.glob('spotify-data/*.json')

records = []
for f in files:
    with open(f, 'r', encoding='utf-8') as fp:
        data = json.load(fp)
        records.extend(data)

# armar dataframe con solo lo que nos importa
df = pd.DataFrame(records)[['ts', 'ms_played', 'master_metadata_track_name', 
                              'master_metadata_album_artist_name', 
                              'master_metadata_album_album_name']]

# renombrar columnas para no volverse loco
df.columns = ['timestamp', 'ms_played', 'track', 'artist', 'album']

# parsear timestamp y filtrar skips (menos de 30 seg)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df[df['ms_played'] >= 30000]

# sacar nulls (podcasts, episodios, etc)
df = df.dropna(subset=['track', 'artist', 'album'])

print(df.head())

                  timestamp  ms_played                track          artist  \
0 2026-02-14 00:18:52+00:00     178696             Above Us    Beloved More   
1 2026-02-14 00:21:03+00:00     142784              Dayglow         Narnara   
2 2026-02-14 00:23:19+00:00     148000  Until the Sky Opens     Somnolentia   
3 2026-02-14 00:25:58+00:00     171001             Nebulith     Revivo Wide   
4 2026-02-14 00:28:39+00:00     172803               Memora  Edge of Dreams   

                 album  
0             Above Us  
1              Dayglow  
2  Until the Sky Opens  
3             Nebulith  
4               Memora  


In [6]:
from datetime import timezone

cutoff = pd.Timestamp('2022-01-01', tz='UTC')
df_4y = df[df['timestamp'] >= cutoff]

album_stats = df_4y.groupby(['artist', 'album']).agg(
    reproducciones=('track', 'count'),
    tracks_unicas=('track', 'nunique'),
    minutos=('ms_played', lambda x: round(x.sum() / 60000))
).reset_index()

album_stats = album_stats[album_stats['tracks_unicas'] >= 2]
album_stats['score'] = album_stats['minutos'] * album_stats['tracks_unicas']

# Pareto 80%
album_stats = album_stats.sort_values('score', ascending=False)
total_score = album_stats['score'].sum()
album_stats['score_pct'] = album_stats['score'] / total_score
album_stats['score_acum'] = album_stats['score_pct'].cumsum()

# Shift para incluir el álbum que cruza el umbral
top = album_stats[album_stats['score_acum'].shift(1, fill_value=0) < 0.80].copy()

print(f"Álbumes que concentran el 80% del score: {len(top)} de {len(album_stats)}")
print(top.drop(columns=['score_pct', 'score_acum']).to_string(index=False))

Álbumes que concentran el 80% del score: 166 de 1395
                                       artist                                                              album  reproducciones  tracks_unicas  minutos  score
                                        Queen                                                        The Miracle             424             47     1621  76187
                                           U2                                                 Songs Of Surrender             317             40     1246  49840
                                Peter Gabriel                                                                Hit             315             30     1518  45540
                                       Eagles                                        Live From The Forum MMXVIII             278             33     1100  36300
           Noel Gallagher's High Flying Birds                         Back the Way We Came: Vol. 1 (2011 - 2021)             297             30    

In [14]:
import os
from rapidfuzz import fuzz

navidrome_albums = set()

music_path = '/mnt/tank-hdd/media/Music'
for artist in os.listdir(music_path):
    artist_path = os.path.join(music_path, artist)
    if os.path.isdir(artist_path):
        for album in os.listdir(artist_path):
            album_path = os.path.join(artist_path, album)
            if os.path.isdir(album_path):
                navidrome_albums.add((artist, album))

# armar set de navidrome en minúsculas para comparación flexible
navidrome_lower = {(a.lower().strip(), b.lower().strip()) for a, b in navidrome_albums}

# top 50 de spotify
top50 = album_stats.sort_values('score', ascending=False).head(50).copy()

def esta_en_navidrome(artist, album, navidrome_albums, threshold=60):
    for nav_artist, nav_album in navidrome_albums:
        if fuzz.ratio(artist.lower().strip(), nav_artist.lower().strip()) > threshold:
            if fuzz.ratio(album.lower().strip(), nav_album.lower().strip()) > threshold:
                return True
    return False

top50['en_navidrome'] = top50.apply(
    lambda r: esta_en_navidrome(r['artist'], r['album'], navidrome_albums),
    axis=1
)

faltantes = top50[top50['en_navidrome'] == False][['artist', 'album', 'score', 'reproducciones']]
print(f"Faltan {len(faltantes)} álbumes del top 50:")
print(faltantes.to_string(index=False))

Faltan 36 álbumes del top 50:
                                       artist                                                              album  score  reproducciones
                                           U2                                                 Songs Of Surrender  12680             317
                                Peter Gabriel                                                                Hit   9450             315
                                       Eagles                                        Live From The Forum MMXVIII   9174             278
           Noel Gallagher's High Flying Birds                         Back the Way We Came: Vol. 1 (2011 - 2021)   8910             297
                                     Coldplay                                                      Everyday Life   6080             380
                                           U2                                                       Achtung Baby   5966             157
                  

In [6]:
track_stats = df_4y.groupby(['artist', 'album', 'track']).agg(
    reproducciones=('track', 'count'),
    minutos=('ms_played', lambda x: round(x.sum() / 60000))
).reset_index()

# Pareto 80% por reproducciones
track_stats = track_stats.sort_values('reproducciones', ascending=False)
total_repr = track_stats['reproducciones'].sum()
track_stats['repr_pct'] = track_stats['reproducciones'] / total_repr
track_stats['repr_acum'] = track_stats['repr_pct'].cumsum()

top_tracks = track_stats[track_stats['repr_acum'].shift(1, fill_value=0) < 0.70].copy()

print(f"Tracks que concentran el 70% de las reproducciones: {len(top_tracks)} de {len(track_stats)}")
print(top_tracks.drop(columns=['repr_pct', 'repr_acum']).to_string(index=False))

Tracks que concentran el 70% de las reproducciones: 2298 de 10360
                                               artist                                                                                                                                                                         album                                                                                                                                                                         track  reproducciones  minutos
                                          Duran Duran                                                                                                                                                                   Duran Duran                                                                                                                                                                Ordinary World             100      538
                   Noel Gallagher's High Flying Birds                           

In [9]:
albums_total = df.groupby(['artist', 'album']).size().reset_index(name='reproducciones')
print(f"Álbumes distintos escuchados desde siempre: {len(albums_total)}")

Álbumes distintos escuchados desde siempre: 8494
